# Corners on Euro 2024 — Data Exploration

**Notebook 1 of the first analysis.** End state: you understand what the StatsBomb open data and 360 freeze frames look like for corner-resulting headed shots in Euro 2024.

**What this notebook is not:** it is not the analysis. It is not where you decide what the defensive structure features are. It is not where you build the `football_analytics/` package. Resist all three urges. If you find yourself extracting reusable functions, stop and put a TODO comment instead — the refactor happens in notebook 2.

**Stop condition for this notebook.** You can answer all of these from memory by the end:

1. How many corners are in Euro 2024 in the open data?
2. How many of those produce shots? How many of those shots are headers?
3. What does a 360 freeze frame at the moment of a corner-resulting headed shot actually look like as a data structure?
4. What is the field of view limitation in 360 data, and how often does it matter for corner shot situations?
5. What's the conversion rate of corner-resulting headed shots vs the StatsBomb xG estimate for those same shots?

If you cannot answer all five by the end of this notebook, you have not finished the notebook. If you can answer them after two days, you have finished it and you should move on regardless of whether the notebook "feels" complete.

## Setup

In [ ]:
# Standard imports for this project.
# statsbombpy is the main loader. mplsoccer handles pitch visuals.
# pandas + numpy for the obvious reasons.
#
# Tip: if you haven't already, `uv add statsbombpy mplsoccer pandas numpy matplotlib`
# and `uv add --dev nbstripout` then `nbstripout --install` in the repo root
# so notebook diffs stay clean.

# imports go here


## 1. Find Euro 2024 in the StatsBomb competitions table

The first thing to verify: Euro 2024 is actually in the open data and you have the competition_id / season_id pair. Don't skip this. The IDs are not stable across releases and hardcoding the wrong pair will silently give you a different tournament.

In [ ]:
# Hint: statsbombpy.sb.competitions() returns a DataFrame of all available competitions.
# Filter to the row where competition_name contains "UEFA Euro" and season_name is "2024".
# Pull out competition_id and season_id and store them in named variables you'll reuse.
#
# Sanity check: print the row. Confirm match_available_360 is True for this competition.
# If it isn't, the rest of the notebook is impossible and you need to stop and figure out why.


## 2. Pull the match list and one match's events

Before you load the full tournament, load one match and look at it. The structure of StatsBomb event data is non-obvious and you should know what's in it before you start filtering across 51 matches.

In [ ]:
# Hint: sb.matches(competition_id, season_id) gives you the match list.
# Pick one match (any one) and load its events with sb.events(match_id).
#
# Look at:
#   events.columns         (how many columns are there? what's actually populated?)
#   events['type'].value_counts()    (what event types exist?)
#   events['play_pattern'].value_counts()    (this is your set piece flag)
#
# The play_pattern values include "From Corner", "From Free Kick", "From Throw In", etc.
# Confirm this is true before you rely on the exact string "From Corner" later.


In [ ]:
# Find the corner *events* in this match.
#
# A corner is a type=Pass with pass.type="Corner".
# Don't confuse this with play_pattern="From Corner", which marks everything in the
# sequence that *follows* a corner, not the corner itself.
#
# Tip: this distinction matters a lot for what you're doing. The corner *delivery*
# is one event. The resulting shot (if any) is a downstream event with
# play_pattern="From Corner". You'll want to link them.
#
# Question to answer: how many corners are in this match? How many resulted in a shot?


## 3. Filter to corner-resulting headed shots across all Euro 2024

Now scale up. The goal of this section is one DataFrame with one row per corner-resulting headed shot, with the columns you'll need to merge in freeze frame data next.

**Watch for these gotchas:**

- Some shots have `play_pattern="From Corner"` but are way downstream of the actual corner (e.g. the team kept possession, recycled, and shot 20 seconds later). Decide whether you want those or only "first shot from the sequence." There's no right answer but you have to make a choice and document it.
- Headers in StatsBomb are coded as `shot.body_part = "Head"`. Confirm the exact spelling.
- A shot from a corner kick that the taker takes themselves (direct corner) is rare but real. Decide how to handle.
- Penalties from a corner? Doesn't happen, but second-phase shots after a foul in the box do. Are those "from corner"? StatsBomb says yes via play_pattern. Your call whether to include them.

In [ ]:
# Loop over matches, accumulate events, filter to headed shots with play_pattern="From Corner".
#
# Tip: sb.events takes a single match_id. The pythonic version is a list comprehension
# with pd.concat, but be aware this loads ~50 matches sequentially over HTTP and will
# take a few minutes. The first time, just run it and walk away. Cache the result to
# a parquet file in data/ (gitignored).
#
# At minimum, keep columns: match_id, id (event id), index, period, timestamp, minute,
# team, player, location, shot.outcome, shot.statsbomb_xg, shot.body_part,
# play_pattern, possession, possession_team.
#
# Question to answer: how many corner-resulting headed shots are there in Euro 2024?
# Is the count plausible given there are 51 matches and ~10 corners per match?


## 4. Pull the 360 freeze frame for each of those shots

**This is the part that's harder than it looks.** Read this paragraph before you write any code:

StatsBomb 360 data is provided per-match, separately from the events. Each event has a `360_freeze_frame` field that's a list of dicts — each dict is one tracked player with `location`, `teammate` (bool, are they on the same team as the actor?), `actor` (bool, are they *the* actor?), and `keeper` (bool). The freeze frame is captured at the moment of the event.

**Critical limitations to internalize now:**

1. **Not all players are in the frame.** The 360 data only tracks players in the broadcast camera's field of view at the moment. For a corner, this means the camera is usually focused on the box, so you usually get most defenders and most attackers in the box — but not always. Frames with fewer than ~14 players are flagged via `visible_area`.
2. **The `visible_area` polygon tells you which parts of the pitch the camera sees.** A player outside the visible area is *not absent* — they're just untracked. Be careful not to treat absence as evidence of absence.
3. **Coordinates are in StatsBomb pitch coordinates** (120 long, 80 wide, attacking direction left to right by convention but verify per event).

**The thing that will trip you up:** for the shot event, the freeze frame is at the moment of the shot, not at the moment of the corner delivery. Those are different moments. Defensive structure at the moment of the shot is *post-contest* — the defenders may have already lost their assignments. Defensive structure at the moment of the corner *delivery* (the corner kick pass event) is what you actually want for predicting outcomes.

So you need to link each corner-resulting shot back to its originating corner kick event, and pull the freeze frame from *the corner kick*, not from the shot. This is a real piece of work.

**Verify first:** before you build anything on top of this assumption, confirm that StatsBomb attaches 360 frames to corner *pass* events. If they don't, you'll need to use the closest preceding tracked event with a frame, and the whole linking step changes.

In [ ]:
# Step 1: get all events with 360 data for the matches in scope.
# sb.events(match_id) actually does include 360_freeze_frame when fmt="json" — confirm
# by checking events.columns after loading one match. If statsbombpy is hiding it by
# default, you may need to pull the raw 360 jsons from the open-data github repo:
# https://github.com/statsbomb/open-data/tree/master/data/three-sixty
#
# Tip: don't fight statsbombpy if it's awkward. Just pull the JSONs directly with requests.
# The 360 file per match is data/three-sixty/{match_id}.json, structured as a list of
# {event_uuid, visible_area, freeze_frame} dicts. Merge on event_uuid == event.id.


In [ ]:
# Step 2: link each corner-resulting shot back to its originating corner kick.
#
# Within a single match's event stream sorted by (period, timestamp), look backwards
# from each headed shot with play_pattern="From Corner" to find the most recent
# type=Pass, pass.type=Corner event. That's the originating delivery.
#
# Edge case: if a shot is more than ~15 seconds after the corner, the link is probably
# spurious (possession was lost and regained). Decide your time threshold and document it.
#
# Edge case: possession_id should be the same for the corner and the resulting shot in
# the simple case. Use that as a sanity check.
#
# Output: a DataFrame with one row per corner-resulting headed shot, with columns for
# both the shot event_id and the originating corner event_id.


In [ ]:
# Step 3: attach the freeze frame from the *corner event* (not the shot event) to each row.
#
# Question to answer: of the corner-resulting headed shots, how many have a 360 freeze
# frame attached at the moment of the corner delivery? Is it 100%? If not, why not?
# (Hint: 360 coverage isn't 100% even in competitions that have it.)


## 5. Visualize one freeze frame

Pick one corner. Plot it. This is the thing that will tell you whether you actually understand the data — if you can produce a pitch plot showing the corner taker's position, the ball location, the attackers, the defenders, and the keeper, all in the right places, you've understood the data structure. If the plot looks wrong, your filtering or your coordinate handling is wrong.

**Do not try to make this beautiful.** Default mplsoccer styling is fine. The goal is to see whether your data makes sense, not to publish.

In [ ]:
# Tip: mplsoccer.Pitch(pitch_type='statsbomb') gives you the right coordinate system.
#
# Plot:
#   - the pitch (probably just the attacking half or final third for visibility)
#   - the ball location (from the corner kick event's location field)
#   - all freeze_frame players, colored by teammate vs opponent
#   - the keeper differently (different marker or color)
#   - optionally the visible_area polygon, to see what the camera covered
#
# Title with: match, minute, team taking the corner, eventual shot outcome.
#
# Plot 3-5 different corners. Spot anything weird? Defenders in implausible positions,
# missing players, attacking team in the wrong half? Anything weird is worth investigating
# before you trust the data.


## 6. Headline numbers: conversion vs xG

This is the original Q3 question, narrowed to your subset: what's the conversion rate of corner-resulting headed shots, and how does it compare to StatsBomb's xG estimate for those shots?

Don't overthink this. You're not building an xG model. You're checking whether the existing xG model is well-calibrated for this specific subset, because if it's *not*, that's itself a finding — and it tells you whether "residual = my model could do better" is a sensible target for the actual analysis.

In [ ]:
# Compute:
#   - total corner-resulting headed shots in Euro 2024
#   - of those, how many were goals (shot.outcome == "Goal")
#   - observed conversion rate = goals / shots
#   - mean StatsBomb xG across these shots
#   - is there a systematic gap? In which direction?
#
# Tip: with maybe 200-300 corner-headed shots in a single tournament, the conversion
# rate has wide error bars. Compute a binomial 95% CI around your observed rate.
# If the mean xG is inside the CI, you can't claim miscalibration from this data alone.
#
# This is a worked example of why this kind of question needs more than one tournament
# of data to answer well — flag it in your notes for the writeup limitations section.


## 7. What you've learned and where it goes next

Before closing this notebook, write three short paragraphs in the cell below:

1. **What does this data actually contain?** Three sentences max.
2. **What surprised you?** (There should be something. If there isn't, you didn't look hard enough.)
3. **What's the smallest interesting question this data can answer?** This is the seed for notebook 2 and the analysis proper.

Then copy these three paragraphs into `writeup.md` under a `## Data exploration notes` section. The writeup is built as you go, not at the end.

_Your three paragraphs here._